# Mechanical mode comb: unequal couplings (archived)

Heterodyne spectra $S_{het}(\omega)$ when the beads' couplings are unequal, for the
supplementary information of the mechanical-mode-comb work: one bead with $g_y/g_x$ from 0.1
to 1; 5 and 10 beads with the same ratios ($g_x/2\pi$ = −40 and −30 kHz); and 5 beads whose
couplings increase, decrease or are random (sorted and unsorted), in the 'equal spacing' and
'pairs' Ω models. Each figure stacks the spectra vertically, one per coupling set.

**Runtime:** several minutes (the 10-bead cells dominate).

**Updated Nov 2026** to current levisim:

- the toy parameter `'gs'` (per-dimension couplings) → `'gjn_dict'`, its name since Dec 2025.
  Today's `MultiTrap` ignores the old key, so without the rename every coupling would silently be 0;
- Ω model `'uniform'` → `'equal spacing'`, the same model under its current name (Oct 2025);
- `calculate_heterodyne_spectrum` → `heterodyne_spectrum`; the inline xmgrace-style rcParams →
  `set_plt_rcParams_to_xmgrace_style()` (the same settings);
- in the last three cells the coupling lists are magnitudes and are now applied as negative
  couplings: today's `Dimension` asserts $g \le 0$ for x and y (flipping the sign of every coupling
  is a symmetry, so the heterodyne spectra are unchanged);
- the random couplings are seeded (`random.seed(0)`), so they differ from the unseeded draws in
  the original run;
- three scratch cells (`kHz·2π = 10`, `kHz·2π`, `π· = 10`) were removed.

# 1 bead

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

num_traps, num_axes = 1, 2
Ntot = num_traps * num_axes

def make_g_dict(gx=None, gy=None, num_traps=num_traps):
    g_dict = {}
    for ii in range(num_traps):
        g_dict[f'x{ii + 1}'] = gx
        g_dict[f'y{ii + 1}'] = gy
    return g_dict

big_g = -40*kHz

gs_dicts = [
    make_g_dict(gx=big_g, gy=0.10 * big_g),
    make_g_dict(gx=big_g, gy=0.25 * big_g),
    make_g_dict(gx=big_g, gy=0.50 * big_g),
    make_g_dict(gx=big_g, gy=0.75 * big_g),
    make_g_dict(gx=big_g, gy=0.90 * big_g),
    make_g_dict(gx=big_g, gy=1.00 * big_g),
]

descs = [
    f'g$_y$/g$_x$ = 0.10',
    f'g$_y$/g$_x$ = 0.25',
    f'g$_y$/g$_x$ = 0.50',
    f'g$_y$/g$_x$ = 0.75',
    f'g$_y$/g$_x$ = 0.90',
    f'g$_y$/g$_x$ = 1.00',
]

# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )
for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'pairs',
        'Ω_first': 250*kHz,
        'Ω_intrapair_gap': 15*kHz,
        'Ω_interpair_gap': 15*kHz,
        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 200, 310, 0.01
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='purple',      # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; g$_x/2π$ = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()


# 5 beads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

num_traps, num_axes = 5, 2
Ntot = num_traps * num_axes

def make_g_dict(gx=None, gy=None, num_traps=num_traps):
    g_dict = {}
    for ii in range(num_traps):
        g_dict[f'x{ii + 1}'] = gx
        g_dict[f'y{ii + 1}'] = gy
    return g_dict

big_g = -40*kHz

gs_dicts = [
    make_g_dict(gx=big_g, gy=0.10 * big_g),
    make_g_dict(gx=big_g, gy=0.25 * big_g),
    make_g_dict(gx=big_g, gy=0.50 * big_g),
    make_g_dict(gx=big_g, gy=0.75 * big_g),
    make_g_dict(gx=big_g, gy=0.90 * big_g),
    make_g_dict(gx=big_g, gy=1.00 * big_g),
]

descs = [
    f'g$_y$/g$_x$ = 0.10',
    f'g$_y$/g$_x$ = 0.25',
    f'g$_y$/g$_x$ = 0.50',
    f'g$_y$/g$_x$ = 0.75',
    f'g$_y$/g$_x$ = 0.90',
    f'g$_y$/g$_x$ = 1.00',
]

# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )
for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'pairs',
        'Ω_first': 325*kHz,
        'Ω_interpair_gap': 1.5*kHz,
        'Ω_interpair_gap': 15*kHz,
        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 200, 330, 0.1
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='blue',        # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; g$_x/2π$ = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()


# 10 beads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0


num_traps, num_axes = 10, 2
Ntot = num_traps * num_axes

def make_g_dict(gx=None, gy=None, num_traps=num_traps):
    g_dict = {}
    for ii in range(num_traps):
        g_dict[f'x{ii + 1}'] = gx
        g_dict[f'y{ii + 1}'] = gy
    return g_dict

big_g = -40*kHz

gs_dicts = [
    make_g_dict(gx=big_g, gy=0.10 * big_g),
    make_g_dict(gx=big_g, gy=0.25 * big_g),
    make_g_dict(gx=big_g, gy=0.50 * big_g),
    make_g_dict(gx=big_g, gy=0.75 * big_g),
    make_g_dict(gx=big_g, gy=0.90 * big_g),
    make_g_dict(gx=big_g, gy=1.00 * big_g),
]

descs = [
    f'g$_y$/g$_x$ = 0.10',
    f'g$_y$/g$_x$ = 0.25',
    f'g$_y$/g$_x$ = 0.50',
    f'g$_y$/g$_x$ = 0.75',
    f'g$_y$/g$_x$ = 0.90',
    f'g$_y$/g$_x$ = 1.00',
]


# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )
for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'pairs',
        'Ω_first': 350*kHz,
        'Ω_interpair_gap': 1.5*kHz,
        'Ω_interpair_gap': 15*kHz,
        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 205, 355, 0.05
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='red',         # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; g$_x/2π$ = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()


# 10 beads (gx = -30 kHz)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0


num_traps, num_axes = 10, 2
Ntot = num_traps * num_axes

def make_g_dict(gx=None, gy=None, num_traps=num_traps):
    g_dict = {}
    for ii in range(num_traps):
        g_dict[f'x{ii + 1}'] = gx
        g_dict[f'y{ii + 1}'] = gy
    return g_dict

big_g = -30*kHz

gs_dicts = [
    make_g_dict(gx=big_g, gy=0.10 * big_g),
    make_g_dict(gx=big_g, gy=0.25 * big_g),
    make_g_dict(gx=big_g, gy=0.50 * big_g),
    make_g_dict(gx=big_g, gy=0.75 * big_g),
    make_g_dict(gx=big_g, gy=0.90 * big_g),
    make_g_dict(gx=big_g, gy=1.00 * big_g),
]

descs = [
    f'g$_y$/g$_x$ = 0.10',
    f'g$_y$/g$_x$ = 0.25',
    f'g$_y$/g$_x$ = 0.50',
    f'g$_y$/g$_x$ = 0.75',
    f'g$_y$/g$_x$ = 0.90',
    f'g$_y$/g$_x$ = 1.00',
]


# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )
for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'pairs',
        'Ω_first': 350*kHz,
        'Ω_interpair_gap': 1.5*kHz,
        'Ω_interpair_gap': 15*kHz,
        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 200, 355, 0.1
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='blue',        # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; g$_x/2π$ = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()


# 5 beads (Random)

In [ ]:
import random

# seeded, so the random couplings (and hence the figures) are reproducible
random.seed(0)
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

# g_list holds coupling magnitudes in kHz: x and y couplings are negative in levisim (Dimension
# asserts g <= 0), and flipping the sign of every coupling leaves the light spectra unchanged
def make_g_dict(gx=None, gy=None, g_list=None):
    if g_list is not None:
        return {
            'x1': -g_list[0] * kHz,
            'y1': -g_list[1] * kHz,

            'x2': -g_list[2] * kHz,
            'y2': -g_list[3] * kHz,

            'x3': -g_list[4] * kHz,
            'y3': -g_list[5] * kHz,

            'x4': -g_list[6] * kHz,
            'y4': -g_list[7] * kHz,

            'x5': -g_list[8] * kHz,
            'y5': -g_list[9] * kHz,

            # 'x6': -g_list[10] * kHz,
            # 'y6': -g_list[11] * kHz,

            # 'x7': -g_list[12] * kHz,
            # 'y7': -g_list[13] * kHz,

            # 'x8': -g_list[14] * kHz,
            # 'y8': -g_list[15] * kHz,

            # 'x9': -g_list[16] * kHz,
            # 'y9': -g_list[17] * kHz,

            # 'x10': -g_list[18] * kHz,
            # 'y10': -g_list[19] * kHz,
        }

num_traps, num_axes = 5, 2
Ntot = num_traps * num_axes

big_g_int = 40
big_g = -big_g_int*kHz

increasing = np.array(range(1, Ntot + 1)) * (big_g_int // Ntot)
decreasing = increasing[::-1]

random_numbers = random.sample(range(1, big_g_int), Ntot)
increasing_random = sorted(random_numbers)
decreasing_random = increasing_random[::-1]

gs_dicts = [
    make_g_dict(g_list=increasing),
    make_g_dict(g_list=decreasing),
    make_g_dict(g_list=random_numbers),
    make_g_dict(g_list=increasing_random),
    make_g_dict(g_list=decreasing_random),
]

descs_with_values = [
    f'increasing $g_j$ = {increasing[:Ntot]} kHz',
    f'decreasing $g_j$ = {decreasing[:Ntot]} kHz',
    f'random $g_j$ = {random_numbers[:Ntot]} kHz',
    f'inc. random $g_j$ = {increasing_random[:Ntot]} kHz',
    f'dec. random $g_j$ = {decreasing_random[:Ntot]} kHz',
]
print(descs_with_values)

descs = [
    f'inc. uniformly $g_j$',
    f'dec. uniformly $g_j$',
    f'random $g_j$',
    f'inc. random $g_j$',
    f'dec. random $g_j$',
]


# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )

for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
        'Ω_mid': 300*kHz,
        'Ω_span': 100*kHz,

        # 'Ω_model': 'pairs',
        # 'Ω_first': 350*kHz,
        # 'Ω_interpair_gap': 1.5*kHz,
        # 'Ω_interpair_gap': 15*kHz,

        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 245, 355, 0.1
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='blue',        # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; max(g$_j/2π$) = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()

# 5 beads  (pairs, big-small on-off)

In [ ]:
import random

# seeded, so the random couplings (and hence the figures) are reproducible
random.seed(0)
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

# g_list holds coupling magnitudes in kHz: x and y couplings are negative in levisim (Dimension
# asserts g <= 0), and flipping the sign of every coupling leaves the light spectra unchanged
def make_g_dict(gx=None, gy=None, g_list=None):
    if g_list is not None:
        return {
            'x1': -g_list[0] * kHz,
            'y1': -g_list[1] * kHz,

            'x2': -g_list[2] * kHz,
            'y2': -g_list[3] * kHz,

            'x3': -g_list[4] * kHz,
            'y3': -g_list[5] * kHz,

            'x4': -g_list[6] * kHz,
            'y4': -g_list[7] * kHz,

            'x5': -g_list[8] * kHz,
            'y5': -g_list[9] * kHz,

            # 'x6': -g_list[10] * kHz,
            # 'y6': -g_list[11] * kHz,

            # 'x7': -g_list[12] * kHz,
            # 'y7': -g_list[13] * kHz,

            # 'x8': -g_list[14] * kHz,
            # 'y8': -g_list[15] * kHz,

            # 'x9': -g_list[16] * kHz,
            # 'y9': -g_list[17] * kHz,

            # 'x10': -g_list[18] * kHz,
            # 'y10': -g_list[19] * kHz,
        }

num_traps, num_axes = 5, 2
Ntot = num_traps * num_axes

big_g_int = 40
big_g = -big_g_int*kHz

increasing = np.array(range(1, Ntot + 1)) * (big_g_int // Ntot)
decreasing = increasing[::-1]

random_numbers = random.sample(range(1, big_g_int), Ntot)
increasing_random = sorted(random_numbers)
decreasing_random = increasing_random[::-1]

gs_dicts = [
    make_g_dict(g_list=[40, 40, 4, 4, 40, 40, 4, 4, 40, 40]),
    make_g_dict(g_list=[4, 4, 40, 40, 4, 4, 40, 40, 4, 4]),
    make_g_dict(g_list=[20, 20, 10, 10, 20, 20, 10, 10, 20, 20]),
    make_g_dict(g_list=[10, 10, 20, 20, 10, 10, 20, 20, 10, 10]),
]

descs = [
    f'40-4-40 paired',
    f'4-40-4 paired',
    f'20-10-20 paired',
    f'10-20-10 paired',
]


# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )

for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        # 'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
        # 'Ω_mid': 300*kHz,
        # 'Ω_span': 100*kHz,

        'Ω_model': 'pairs',
        'Ω_first': 325*kHz,
        'Ω_interpair_gap': 1.5*kHz,
        'Ω_interpair_gap': 15*kHz,

        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 245, 355, 0.1
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='green',       # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; max(g$_j/2π$) = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
import random

# seeded, so the random couplings (and hence the figures) are reproducible
random.seed(0)
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

# g_list holds coupling magnitudes in kHz: x and y couplings are negative in levisim (Dimension
# asserts g <= 0), and flipping the sign of every coupling leaves the light spectra unchanged
def make_g_dict(gx=None, gy=None, g_list=None):
    if g_list is not None:
        return {
            'x1': -g_list[0] * kHz,
            'y1': -g_list[1] * kHz,

            'x2': -g_list[2] * kHz,
            'y2': -g_list[3] * kHz,

            'x3': -g_list[4] * kHz,
            'y3': -g_list[5] * kHz,

            'x4': -g_list[6] * kHz,
            'y4': -g_list[7] * kHz,

            'x5': -g_list[8] * kHz,
            'y5': -g_list[9] * kHz,

            # 'x6': -g_list[10] * kHz,
            # 'y6': -g_list[11] * kHz,

            # 'x7': -g_list[12] * kHz,
            # 'y7': -g_list[13] * kHz,

            # 'x8': -g_list[14] * kHz,
            # 'y8': -g_list[15] * kHz,

            # 'x9': -g_list[16] * kHz,
            # 'y9': -g_list[17] * kHz,

            # 'x10': -g_list[18] * kHz,
            # 'y10': -g_list[19] * kHz,
        }

num_traps, num_axes = 5, 2
Ntot = num_traps * num_axes

big_g_int = 40
big_g = -big_g_int*kHz

increasing = np.array(range(1, Ntot + 1)) * (big_g_int // Ntot)
decreasing = increasing[::-1]

random_numbers = random.sample(range(1, big_g_int), Ntot)
increasing_random = sorted(random_numbers)
decreasing_random = increasing_random[::-1]

gs_dicts = [
    make_g_dict(g_list=[40, 40, 4, 4, 40, 40, 4, 4, 40, 40]),
    make_g_dict(g_list=[4, 4, 40, 40, 4, 4, 40, 40, 4, 4]),
    make_g_dict(g_list=[20, 20, 10, 10, 20, 20, 10, 10, 20, 20]),
    make_g_dict(g_list=[10, 10, 20, 20, 10, 10, 20, 20, 10, 10]),
]

descs = [
    f'40-4-40 equal',
    f'4-40-4 equal',
    f'20-10-20 equal',
    f'10-20-10 equal',
]


# Create a figure with a mosaic layout - one subplot per item in descs
fig = plt.figure(figsize=(5, 2 * len(descs)))  # Adjust size as needed
mosaic = []
for ii in range(len(descs)):
    mosaic.append([f'plot{ii}'])

ax_dict = fig.subplot_mosaic(
    mosaic,
    sharex=True,
    gridspec_kw={'hspace': 0},
    )

for ii, (gs_dict, desc) in enumerate(zip(gs_dicts, descs)):
    print(desc)

    toy_params = {
        'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
        'Ω_mid': 300*kHz,
        'Ω_span': 100*kHz,

        # 'Ω_model': 'pairs',
        # 'Ω_first': 325*kHz,
        # 'Ω_interpair_gap': 1.5*kHz,
        # 'Ω_interpair_gap': 15*kHz,

        'gjn_dict': gs_dict,
        'use_Γ_formula': True,
    }

    cavity_kwargs = {
        'Δ': -450*kHz,
        'κ': 200*kHz,
    }

    gas_kwargs = {
        'P': 1e-7*mbar,
        'T': 300*K,
        'vth': 500, # m/s
    }

    low, high, low_res = 0, 400, 2.5
    hd_a, hd_b, hd_res = 245, 355, 0.1
    ω_range = np.concatenate(
        (
            np.arange(low, hd_a, low_res) * kHz,
            np.arange(hd_a, hd_b, hd_res) * kHz,
            np.arange(hd_b, high, low_res) * kHz,
        ),
    )

    multitrap = MultiTrap(
        num_traps=num_traps,
        Cartesian_axes_per_trap=num_axes,
        ω_range=ω_range,
        cavity_kwargs=cavity_kwargs,
        gas_kwargs=gas_kwargs,
        toy_params=toy_params,
    )
    plot_values = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    ax = ax_dict[f'plot{ii}']
    ax.fill_between(
        multitrap.ω_range / kHz,
        plot_values,
        hatch='.....',           # Dot pattern (adjust '.' count for density)
        edgecolor='green',       # Edge color of dots
        facecolor='none',        # No solid fill (only dots)
        linewidth=1.5,           # No edge line (only dots)
        alpha=0.8,               # Adjust dot opacity
    )
    ax.set_yscale('log')

    ax.set_yticks([0.1, 1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
    ax.set_ylim(1e-1, 0.99999*1e2)
    ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

    # --- Add transparent textbox to each subplot ---
    text_options = {
        's': desc,      # Text content
        'fontsize': 15,         # Font size
        'fontname': 'Arial',    # Font family (to match Grace)
        'bbox': {
            'facecolor': 'white',  # Background color
            'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
            'edgecolor': 'none',   # No border
            'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
        },
        'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
        'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
        'zorder': 100,          # Ensure text is on top
    }

    # Add to subplot 1 (top-left corner)
    ax.text(
        0.2, 0.8,
        # axes-relative coordinates
        transform=ax.transAxes,
        **text_options
    )


fig.suptitle(f'N$_p$={num_traps}; max(g$_j/2π$) = {big_g / kHz:.0f} kHz', fontsize=20)
ax_dict[f'plot{len(descs)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
plt.tight_layout()
plt.show()